In [2]:
# Cell 1 - Mount Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# Cell 2 - Extract VOC dataset
import zipfile, os
os.makedirs('/content/dataset/voc', exist_ok=True)
with zipfile.ZipFile('/content/drive/MyDrive/RD2/voc_dataset.zip', 'r') as z:
    z.extractall('/content/dataset/voc')
print("Done")

Done


In [4]:
# Cell 3 - Install dependencies
!pip install scikit-learn -q

In [5]:
import os
import numpy as np
import torch
import torchvision
from torchvision.models.detection import ssd300_vgg16
from torchvision.models.detection.ssd import SSDClassificationHead
from torchvision.models.detection.anchor_utils import DefaultBoxGenerator
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
from pathlib import Path
from PIL import Image
import xml.etree.ElementTree as ET
from torchvision import transforms
from torchvision.ops import box_iou
import torch.nn as nn

# ─── CONFIG ───────────────────────────────────────────────────────────────────
DATASET_DIR  = "/content/dataset/voc/voc/Smartphone-Detection-2/train"
K_FOLDS      = 5
EPOCHS       = 50
IMG_SIZE     = 300  # SSD300 expects 300x300
NUM_CLASSES  = 3    # background + person + smartphone
RESULTS_DIR  = "/content/results/ssd"
DEVICE       = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
# ──────────────────────────────────────────────────────────────────────────────

os.makedirs(RESULTS_DIR, exist_ok=True)
print(f"Using device: {DEVICE}")

# Roboflow exports numeric labels in VOC format
CLASS_MAP   = {"0": 1, "1": 2}
LABEL_NAMES = {1: "person", 2: "smartphone"}

# ─── DATASET ──────────────────────────────────────────────────────────────────
class VOCDataset(Dataset):
    def __init__(self, image_paths, dataset_dir):
        self.image_paths = image_paths
        self.dataset_dir = Path(dataset_dir)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        xml_path = self.dataset_dir / (Path(img_path).stem + ".xml")

        img = Image.open(img_path).convert("RGB")
        img = img.resize((IMG_SIZE, IMG_SIZE))
        img_tensor = transforms.ToTensor()(img)

        boxes, labels = [], []
        if xml_path.exists():
            tree = ET.parse(xml_path)
            root = tree.getroot()
            orig_w = int(root.find("size/width").text)
            orig_h = int(root.find("size/height").text)
            scale_x = IMG_SIZE / orig_w
            scale_y = IMG_SIZE / orig_h

            for obj in root.findall("object"):
                label = obj.find("name").text.strip()
                if label not in CLASS_MAP:
                    continue
                bbox = obj.find("bndbox")
                xmin = float(bbox.find("xmin").text) * scale_x
                ymin = float(bbox.find("ymin").text) * scale_y
                xmax = float(bbox.find("xmax").text) * scale_x
                ymax = float(bbox.find("ymax").text) * scale_y
                if xmax > xmin and ymax > ymin:
                    boxes.append([xmin, ymin, xmax, ymax])
                    labels.append(CLASS_MAP[label])

        if len(boxes) == 0:
            boxes  = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,),   dtype=torch.int64)
        else:
            boxes  = torch.tensor(boxes,  dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)

        target = {
            "boxes":  boxes,
            "labels": labels,
        }
        return img_tensor, target

def collate_fn(batch):
    return tuple(zip(*batch))

# ─── IoU HELPER ───────────────────────────────────────────────────────────────
def compute_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])
    inter = max(0, x2 - x1) * max(0, y2 - y1)
    area1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    area2 = (box2[2]-box2[0]) * (box2[3]-box2[1])
    union = area1 + area2 - inter
    return inter / union if union > 0 else 0

# ─── EVALUATION ───────────────────────────────────────────────────────────────
def evaluate(model, loader, iou_thresh=0.5):
    model.eval()
    all_tp, all_fp, all_fn = 0, 0, 0
    total_preds = 0
    total_gt    = 0

    with torch.no_grad():
        for images, targets in loader:
            images  = [img.to(DEVICE) for img in images]
            outputs = model(images)

            for output, target in zip(outputs, targets):
                pred_boxes  = output["boxes"].cpu().numpy()
                pred_scores = output["scores"].cpu().numpy()
                pred_labels = output["labels"].cpu().numpy()
                gt_boxes    = target["boxes"].numpy()
                gt_labels   = target["labels"].numpy()

                keep        = pred_scores >= 0.05
                pred_boxes  = pred_boxes[keep]
                pred_labels = pred_labels[keep]
                pred_scores = pred_scores[keep]

                total_preds += len(pred_boxes)
                total_gt    += len(gt_boxes)

                if len(gt_boxes) == 0:
                    all_fp += len(pred_boxes)
                    continue

                if len(pred_boxes) == 0:
                    all_fn += len(gt_boxes)
                    continue

                order       = pred_scores.argsort()[::-1]
                pred_boxes  = pred_boxes[order]
                pred_labels = pred_labels[order]

                matched_gt = set()
                tp, fp = 0, 0

                for pb, pl in zip(pred_boxes, pred_labels):
                    best_iou = 0
                    best_j   = -1
                    for j, (gb, gl) in enumerate(zip(gt_boxes, gt_labels)):
                        if j in matched_gt:
                            continue
                        iou = compute_iou(pb, gb)
                        if iou > best_iou:
                            best_iou = iou
                            best_j   = j
                    if best_iou >= iou_thresh and best_j >= 0:
                        tp += 1
                        matched_gt.add(best_j)
                    else:
                        fp += 1

                fn = len(gt_boxes) - len(matched_gt)
                all_tp += tp
                all_fp += fp
                all_fn += fn

    print(f"  Debug — Preds: {total_preds} | GT: {total_gt} | TP: {all_tp} | FP: {all_fp} | FN: {all_fn}")

    precision = all_tp / (all_tp + all_fp) if (all_tp + all_fp) > 0 else 0
    recall    = all_tp / (all_tp + all_fn) if (all_tp + all_fn) > 0 else 0
    f1        = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# ─── MAIN TRAINING LOOP ───────────────────────────────────────────────────────
dataset_dir = Path(DATASET_DIR)
image_files = sorted([
    str(f) for f in dataset_dir.iterdir()
    if f.suffix.lower() in [".jpg", ".jpeg", ".png"]
])
print(f"Total images found: {len(image_files)}")

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold_results = []

for fold, (train_idx, val_idx) in enumerate(kf.split(image_files), 1):
    print(f"\n{'='*40}")
    print(f"FOLD {fold}/{K_FOLDS}")
    print(f"  Train: {len(train_idx)} images")
    print(f"  Test:  {len(val_idx)} images")
    print(f"{'='*40}")

    train_paths = [image_files[i] for i in train_idx]
    val_paths   = [image_files[i] for i in val_idx]

    train_dataset = VOCDataset(train_paths, dataset_dir)
    val_dataset   = VOCDataset(val_paths,   dataset_dir)

    train_loader = DataLoader(train_dataset, batch_size=4,
                              shuffle=True,  collate_fn=collate_fn)
    val_loader   = DataLoader(val_dataset,   batch_size=4,
                              shuffle=False, collate_fn=collate_fn)

    # Load pretrained SSD300 and replace classification head
    model = ssd300_vgg16(weights="DEFAULT")
    num_anchors = model.anchor_generator.num_anchors_per_location()
    in_channels = [512, 1024, 512, 256, 256, 256]
    model.head.classification_head = SSDClassificationHead(
        in_channels=in_channels,
        num_anchors=num_anchors,
        num_classes=NUM_CLASSES
    )
    model.to(DEVICE)

    optimizer = torch.optim.SGD(
        model.parameters(), lr=0.001,
        momentum=0.9, weight_decay=0.0005
    )
    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer, step_size=15, gamma=0.1
    )

    # Training
    for epoch in range(1, EPOCHS + 1):
        model.train()
        epoch_loss = 0
        for images, targets in train_loader:
            images  = [img.to(DEVICE) for img in images]
            targets = [{k: v.to(DEVICE) for k, v in t.items()} for t in targets]
            try:
                loss_dict = model(images, targets)
                loss = sum(loss_dict.values())
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            except Exception as e:
                continue
        scheduler.step()
        if epoch % 10 == 0:
            print(f"  Epoch {epoch}/{EPOCHS} — Loss: {epoch_loss/len(train_loader):.4f}")

    # Evaluate
    precision, recall, f1 = evaluate(model, val_loader)
    fold_results.append({
        "fold":      fold,
        "precision": precision,
        "recall":    recall,
        "f1":        f1
    })

    print(f"\nFold {fold} Results:")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1:        {f1:.4f}")

    # Save model
    torch.save(model.state_dict(),
               f"{RESULTS_DIR}/ssd_fold{fold}.pt")

# ─── FINAL SUMMARY ────────────────────────────────────────────────────────────
print(f"\n{'='*40}")
print("SSD K-FOLD FINAL RESULTS")
print(f"{'='*40}")
for r in fold_results:
    print(f"Fold {r['fold']}: Precision={r['precision']:.4f} | "
          f"Recall={r['recall']:.4f} | F1={r['f1']:.4f}")

means = {k: np.mean([r[k] for r in fold_results])
         for k in ["precision", "recall", "f1"]}
stds  = {k: np.std([r[k]  for r in fold_results])
         for k in ["precision", "recall", "f1"]}

print(f"\nMean ± Std across {K_FOLDS} folds:")
for k in ["precision", "recall", "f1"]:
    print(f"  {k}: {means[k]:.4f} ± {stds[k]:.4f}")

# Save to Drive
summary_path = "/content/drive/MyDrive/RD2/ssd_kfold_results.txt"
with open(summary_path, "w") as f:
    for r in fold_results:
        f.write(f"Fold {r['fold']}: Precision={r['precision']:.4f} | "
                f"Recall={r['recall']:.4f} | F1={r['f1']:.4f}\n")
    for k in ["precision", "recall", "f1"]:
        f.write(f"Mean {k}: {means[k]:.4f} ± {stds[k]:.4f}\n")

print(f"\nResults saved to Google Drive at RD2/ssd_kfold_results.txt")

Using device: cuda
Total images found: 157

FOLD 1/5
  Train: 125 images
  Test:  32 images
Downloading: "https://download.pytorch.org/models/ssd300_vgg16_coco-b556d3b4.pth" to /root/.cache/torch/hub/checkpoints/ssd300_vgg16_coco-b556d3b4.pth


100%|██████████| 136M/136M [00:00<00:00, 245MB/s]


  Epoch 10/50 — Loss: 0.5209
  Epoch 20/50 — Loss: 0.1975
  Epoch 30/50 — Loss: 0.1726
  Epoch 40/50 — Loss: 0.1643
  Epoch 50/50 — Loss: 0.1668
  Debug — Preds: 64 | GT: 64 | TP: 63 | FP: 1 | FN: 1

Fold 1 Results:
  Precision: 0.9844
  Recall:    0.9844
  F1:        0.9844

FOLD 2/5
  Train: 125 images
  Test:  32 images
  Epoch 10/50 — Loss: 0.5610
  Epoch 20/50 — Loss: 0.1983
  Epoch 30/50 — Loss: 0.1765
  Epoch 40/50 — Loss: 0.1900
  Epoch 50/50 — Loss: 0.1720
  Debug — Preds: 65 | GT: 64 | TP: 64 | FP: 1 | FN: 0

Fold 2 Results:
  Precision: 0.9846
  Recall:    1.0000
  F1:        0.9922

FOLD 3/5
  Train: 126 images
  Test:  31 images
  Epoch 10/50 — Loss: 0.4514
  Epoch 20/50 — Loss: 0.1899
  Epoch 30/50 — Loss: 0.1696
  Epoch 40/50 — Loss: 0.1630
  Epoch 50/50 — Loss: 0.1626
  Debug — Preds: 67 | GT: 62 | TP: 62 | FP: 5 | FN: 0

Fold 3 Results:
  Precision: 0.9254
  Recall:    1.0000
  F1:        0.9612

FOLD 4/5
  Train: 126 images
  Test:  31 images
  Epoch 10/50 — Loss: 0.4

In [6]:
import os

paths_to_check = [
    "/content/results/yolo/fold_1/weights/best.pt",
    "/content/results/fasterrcnn/fasterrcnn_fold5.pt",
    "/content/results/ssd/ssd_fold2.pt"
]

for p in paths_to_check:
    exists = os.path.exists(p)
    print(f"{'✅' if exists else '❌'} {p}")

❌ /content/results/yolo/fold_1/weights/best.pt
❌ /content/results/fasterrcnn/fasterrcnn_fold5.pt
✅ /content/results/ssd/ssd_fold2.pt


In [7]:
import shutil, os

os.makedirs('/content/drive/MyDrive/RD2/models/ssd', exist_ok=True)

for fold in range(1, 6):
    src = f'/content/results/ssd/ssd_fold{fold}.pt'
    dst = f'/content/drive/MyDrive/RD2/models/ssd/ssd_fold{fold}.pt'
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'✅ Saved fold {fold}')
    else:
        print(f'❌ Not found: {fold}')

✅ Saved fold 1
✅ Saved fold 2
✅ Saved fold 3
✅ Saved fold 4
✅ Saved fold 5
